# Kube SRE Gym — GRPO Training on Colab

**OpenEnv Hackathon Submission** | [HF Space](https://openenv-community-kube-sre-gym.hf.space) | [GitHub](https://huggingface.co/spaces/openenv-community/kube-sre-gym)

Train a Kubernetes SRE agent using **GRPO** with HF TRL. The agent learns to diagnose and fix real K8s incidents on a **live GKE cluster** — importing all training logic directly from the `kube_sre_gym` package.

| Component | Detail |
|-----------|--------|
| Environment | [HF Space](https://openenv-community-kube-sre-gym.hf.space) (real GKE cluster) |
| Training | This Colab notebook (T4 / A100 GPU) |
| Model | `Qwen/Qwen3-0.6B` or `Qwen/Qwen3-1.7B` + LoRA |
| Framework | HF TRL v0.29+ GRPO with vLLM backend |

## 1. Install Dependencies

Install the `kube_sre_gym` package (includes training utils, client, models) and TRL with vLLM backend.

In [ ]:
!pip install -q \
    "trl[vllm]>=0.29.0" \
    "vllm>=0.11.0" \
    "peft" \
    "transformers" \
    "datasets" \
    "huggingface_hub>=0.20.0" \
    "matplotlib" \
    "numpy"

# Install kube_sre_gym package (client + training utils from the HF Space repo)
!pip install -q "openenv-kube_sre_gym[train] @ git+https://huggingface.co/spaces/openenv-community/kube-sre-gym" 

## 2. Configuration

Set the HF Space URL, model, and training hyperparameters. Add `HF_TOKEN` to Colab Secrets (key icon in sidebar).

In [ ]:
import os

# --- HuggingFace token ---
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except (ImportError, KeyError, ModuleNotFoundError):
    if "HF_TOKEN" not in os.environ:
        print("WARNING: HF_TOKEN not found. Set it in Colab Secrets or as env var.")

# --- Environment server (HF Space) ---
ENV_URL = "https://openenv-community-kube-sre-gym.hf.space"

# --- Model ---
MODEL_ID = "Qwen/Qwen3-0.6B"  # or "Qwen/Qwen3-1.7B" for stronger model
HUB_REPO = "openenv-community/k8s-sre-agent-qwen3-0.6b"

# --- Training hyperparameters ---
NUM_EPISODES = 50
NUM_GENERATIONS = 8
MAX_TURNS = 15

print(f"Environment : {ENV_URL}")
print(f"Model       : {MODEL_ID}")
print(f"Episodes    : {NUM_EPISODES}")
print(f"Generations : {NUM_GENERATIONS}")

## 3. Smoke Test — Verify Environment Connectivity

Connect to the HF Space, reset the cluster (injects faults), and run one command to confirm round-trip works.

In [ ]:
from kube_sre_gym import KubeSreGymEnv, KubeSreGymAction

print(f"Connecting to {ENV_URL} ...")

with KubeSreGymEnv(base_url=ENV_URL) as env:
    # Reset — injects faults into the live GKE cluster
    reset_result = env.reset()
    obs = reset_result.observation
    print("Connected!\n")
    print(f"Alert:\n{obs.command_output[:300]}")
    print(f"\nCluster status:\n{obs.cluster_status_summary[:500]}")

    # Run one kubectl command to verify
    step_result = env.step(KubeSreGymAction(command="kubectl get pods -A"))
    step_obs = step_result.observation
    print(f"\n--- Smoke test step (reward={step_result.reward:.2f}) ---")
    print(step_obs.command_output[:600])

print("\nSmoke test passed. Environment is ready for training.")

## 4. Import Training Utilities from Package

All training logic (system prompt, rollout function, reward functions, helpers) is imported
directly from `kube_sre_gym.train` — the same code used for H100 training. No duplication.

In [ ]:
import csv
import logging
from datetime import datetime
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
from trl.experimental.openenv import generate_rollout_completions

# Import ALL training utilities from the kube_sre_gym package
from kube_sre_gym.train import (
    SYSTEM_PROMPT,
    rollout_once,
    format_observation,
    format_history,
    parse_commands,
    apply_chat_template,
    reward_total,
    reward_diagnosis,
    reward_fix,
    plot_rewards,
    patch_trl_vllm_compat,
)

# Apply TRL/vLLM compatibility patches
patch_trl_vllm_compat()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

print("System prompt (first 200 chars):")
print(SYSTEM_PROMPT[:200])
print("...")
print(f"\nImported: rollout_once, format_observation, parse_commands, reward_total, etc.")

## 5. GRPO Training Setup

Configure tokenizer, environment connection, reward logging, and the GRPO trainer — all using functions imported from the package.

In [ ]:
# ---- Tokenizer ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ---- Environment ----
env = KubeSreGymEnv(base_url=ENV_URL)

# ---- Dataset (each entry triggers one episode) ----
dataset = Dataset.from_dict({"prompt": ["Diagnose and fix this Kubernetes incident."] * NUM_EPISODES})

# ---- Output directory ----
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = Path(f"outputs/k8s-sre-grpo-{timestamp}")
output_dir.mkdir(parents=True, exist_ok=True)

# ---- Reward CSV logger ----
reward_log_path = output_dir / "reward_log.csv"
episode_counter = [0]
all_rewards = []

with open(reward_log_path, "w", newline="") as f:
    csv.writer(f).writerow(["episode", "total_reward", "diagnosis_reward", "fix_reward", "timestamp"])

def log_episode(total_r, diag_r, fix_r):
    episode_counter[0] += 1
    all_rewards.append(total_r)
    with open(reward_log_path, "a", newline="") as f:
        csv.writer(f).writerow([episode_counter[0], total_r, diag_r, fix_r, datetime.now().isoformat()])
    last10 = all_rewards[-10:]
    logger.info(f"Episode {episode_counter[0]}: reward={total_r:.2f} | "
                f"mean={sum(all_rewards)/len(all_rewards):.2f}, mean(10)={sum(last10)/len(last10):.2f}")

# ---- Rollout function (uses rollout_once from package) ----
def rollout_func(prompts, trainer):
    results = {k: [] for k in ["prompt_ids", "completion_ids", "logprobs",
                                "total_reward", "diagnosis_reward", "fix_reward"]}
    for _ in prompts:
        ep = rollout_once(trainer, env, tokenizer, SYSTEM_PROMPT, MAX_TURNS)
        for k in results:
            results[k].append(ep[k])
        log_episode(ep["total_reward"], ep["diagnosis_reward"], ep["fix_reward"])
    return results

print(f"Training setup complete. Output: {output_dir}")

In [ ]:
# ---- GRPO Config ----
grpo_config = GRPOConfig(
    use_vllm=True,
    vllm_mode="colocate",
    output_dir=str(output_dir),
    num_train_epochs=1,
    learning_rate=2e-6,
    gradient_accumulation_steps=8,
    per_device_train_batch_size=1,
    generation_batch_size=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=512,
    logging_steps=1,
    save_strategy="steps",
    save_steps=10,
    temperature=0.8,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_total_limit=3,
)

# ---- LoRA Config ----
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# ---- Create Trainer (reward functions imported from package) ----
trainer = GRPOTrainer(
    model=MODEL_ID,
    processing_class=tokenizer,
    reward_funcs=[reward_total, reward_diagnosis, reward_fix],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
    peft_config=peft_config,
)

print("GRPOTrainer initialized")

## 6. Train!

Launch GRPO training. Each episode: reset cluster → inject faults → agent diagnoses & fixes → LLM judge scores → GRPO gradient update.

In [ ]:
print("Starting GRPO training...")
print(f"  Model       : {MODEL_ID}")
print(f"  Episodes    : {NUM_EPISODES}")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Environment : {ENV_URL}")
print()

try:
    trainer.train()
finally:
    env.close()

trainer.save_model(str(output_dir))
print(f"\nModel saved to {output_dir}")

## 7. Reward Curves

Visualize training progress using `plot_rewards` from the package.

In [ ]:
# Use plot_rewards from the package
plot_rewards(reward_log_path, output_dir / "reward_curve.png")

# Also display inline
import matplotlib.pyplot as plt
from matplotlib.image import imread

img = imread(str(output_dir / "reward_curve.png"))
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(img)
ax.axis("off")
plt.show()

print(f"Reward curve saved to {output_dir / 'reward_curve.png'}")

## 8. Push to Hub (Optional)

Upload the trained LoRA adapter to HuggingFace Hub.

In [ ]:
# Uncomment to push:
# trainer.push_to_hub(repo_id=HUB_REPO)
# print(f"Model pushed to https://huggingface.co/{HUB_REPO}")